In [ ]:
import platform
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

print("Architecture:", platform.machine())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)

In [ ]:
from pathlib import Path

data_folder = Path("../data/raw")

files = list(data_folder.glob("*.mat"))

print("Number of .mat files:", len(files))

for file in files:
    print(file.name)

In [ ]:
from scipy.io import loadmat

sample_file = files[0]

data = loadmat(sample_file)

print("File:", sample_file.name)
print("\nVariables inside the file:")

for key in data.keys():
    print(key)

In [ ]:
print("File:", sample_file.name)
print("Sampling rate:", fs, "Hz")
print("Total samples:", len(de_signal))

In [ ]:
de_keys = [key for key in data.keys() if "DE_time" in key]

print("Drive-end signal variable:", de_keys)

In [ ]:
de_signal = data[de_keys[0]].flatten()

print("Number of samples:", len(de_signal))
print("First 10 acceleration values:")
print(de_signal[:10])

In [ ]:
fs = 12000

time = np.arange(len(de_signal)) / fs

print("Recording duration:", len(de_signal) / fs, "seconds")

In [ ]:
print("Loaded file:", sample_file.name)

rpm_keys = [key for key in data.keys() if "RPM" in key]

print("RPM variable:", rpm_keys)

if rpm_keys:
    print("RPM:", data[rpm_keys[0]].flatten())

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(time, de_signal)

plt.xlabel("Time (s)")
plt.ylabel("Acceleration (g)")
plt.title(f"Drive-End Bearing Vibration — {sample_file.name}")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
duration = 0.1
samples_to_plot = int(fs * duration)

plt.figure(figsize=(12, 4))

plt.plot(
    time[:samples_to_plot],
    de_signal[:samples_to_plot]
)

plt.xlabel("Time (s)")
plt.ylabel("Acceleration (g)")
plt.title(f"Drive-End Bearing Vibration — First 0.1 s — {sample_file.name}")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path

figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)

print("Figures will be saved to:", figures_dir.resolve())

In [ ]:
print("File:", sample_file.name)
print("Sampling rate:", fs, "Hz")
print("Total samples:", len(de_signal))

In [ ]:
from pathlib import Path
from scipy.io import loadmat
import numpy as np

# Select the B007_1 data file
sample_file = Path("../data/raw/B007_1.mat")

# Load MATLAB file
mat = loadmat(sample_file)

# Find the drive-end vibration variable
de_keys = [key for key in mat.keys() if "DE_time" in key]

# Find RPM variable
rpm_keys = [key for key in mat.keys() if "RPM" in key]

# Extract drive-end vibration signal
de_signal = mat[de_keys[0]].flatten()

# Extract RPM
rpm = float(mat[rpm_keys[0]].flatten()[0])

# Sampling rate for this 12 kHz CWRU fault recording
fs = 12000

print("File:", sample_file.name)
print("Drive-end variable:", de_keys[0])
print("RPM variable:", rpm_keys[0])
print("RPM:", rpm)
print("Sampling rate:", fs, "Hz")
print("Total samples:", len(de_signal))
print("Duration:", len(de_signal) / fs, "seconds")

In [ ]:
window_duration = 1.0

window_size = int(fs * window_duration)

num_complete_windows = len(de_signal) // window_size

remaining_samples = len(de_signal) % window_size

print("Sampling rate:", fs, "Hz")
print("Samples per window:", window_size)
print("Total samples:", len(de_signal))
print("Complete 1-second windows:", num_complete_windows)
print("Remaining samples:", remaining_samples)

In [ ]:
windows = []

for i in range(num_complete_windows):
    start = i * window_size
    end = start + window_size
    
    window = de_signal[start:end]
    windows.append(window)

# Convert the list into a NumPy array
windows = np.array(windows)

print("Windows array shape:", windows.shape)
print("Number of windows:", windows.shape[0])
print("Samples in each window:", windows.shape[1])

In [ ]:
window_number = 0

window_time = np.arange(window_size) / fs

plt.figure(figsize=(12, 4))

plt.plot(
    window_time,
    windows[window_number]
)

plt.xlabel("Time (s)")
plt.ylabel("Acceleration (g)")
plt.title("B007_1 — Window 1 — 1 Second")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import skew, kurtosis

def extract_features(window):
    
    # Root Mean Square - overall vibration energy
    rms = np.sqrt(np.mean(window ** 2))
    
    # Standard deviation - spread of vibration amplitudes
    std_dev = np.std(window)
    
    # Largest absolute vibration amplitude
    peak = np.max(np.abs(window))
    
    # Difference between maximum and minimum amplitude
    peak_to_peak = np.ptp(window)
    
    # Asymmetry of amplitude distribution
    skewness = skew(window)
    
    # Impulsiveness / heavy-tailed behaviour
    # fisher=False means normal distribution ≈ 3
    kurt = kurtosis(window, fisher=False)
    
    # Ratio between largest peak and overall vibration level
    crest_factor = peak / rms if rms != 0 else 0
    
    return {
        "RMS": rms,
        "Std_Dev": std_dev,
        "Peak": peak,
        "Peak_to_Peak": peak_to_peak,
        "Skewness": skewness,
        "Kurtosis": kurt,
        "Crest_Factor": crest_factor
    }

In [ ]:
window_1_features = extract_features(windows[0])

print("Features for B007_1 - Window 1")
print("--------------------------------")

for feature, value in window_1_features.items():
    print(f"{feature:15s}: {value:.6f}")

In [ ]:
import pandas as pd

feature_rows = []

for i, window in enumerate(windows):
    
    features = extract_features(window)
    
    # Add information identifying this observation
    features["Window"] = i + 1
    features["Fault_Type"] = "Ball"
    features["Load_HP"] = 1
    features["RPM"] = 1772
    features["Source_File"] = sample_file.name
    
    feature_rows.append(features)

# Convert everything into a DataFrame
b007_features = pd.DataFrame(feature_rows)

print("Feature dataset shape:", b007_features.shape)

b007_features

In [ ]:
from pathlib import Path

raw_data_dir = Path("../data/raw")

mat_files = sorted(raw_data_dir.glob("*.mat"))

print("Number of .mat files:", len(mat_files))
print("\nFiles found:")
print("------------------------------")

for file in mat_files:
    print(file.name)

In [ ]:
feature_columns = [
    "RMS",
    "Std_Dev",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

b007_features[feature_columns].describe()

In [ ]:
from scipy.io import loadmat
import numpy as np
import pandas as pd

qc_rows = []

for file in mat_files:

    mat = loadmat(file)

    # Find drive-end vibration variable
    de_keys = [key for key in mat.keys() if "DE_time" in key]

    # Find RPM variable
    rpm_keys = [key for key in mat.keys() if "RPM" in key]

    if len(de_keys) == 0:
        print("WARNING: No DE signal found in", file.name)
        continue

    # Extract DE signal
    signal = mat[de_keys[0]].flatten()

    # Extract RPM if available
    if len(rpm_keys) > 0:
        rpm_value = float(mat[rpm_keys[0]].flatten()[0])
    else:
        rpm_value = np.nan

    qc_rows.append({
        "File": file.name,
        "DE_Key": de_keys[0],
        "Samples": len(signal),
        "RPM": rpm_value
    })

qc_table = pd.DataFrame(qc_rows)

qc_table

In [ ]:
from scipy.io import loadmat
from scipy.signal import resample_poly
import numpy as np
import pandas as pd

TARGET_FS = 12000
WINDOW_SECONDS = 1
WINDOW_SIZE = TARGET_FS * WINDOW_SECONDS

# Official approximate speeds for each load
rpm_by_load = {
    0: 1797,
    1: 1772,
    2: 1750,
    3: 1730
}


def get_file_info(filename):
    """
    Determine fault type and motor load
    from our standardized filenames.
    """

    name = filename.replace(".mat", "")

    # Fault type

    if name.startswith("Normal"):
        fault_type = "Normal"

    elif name.startswith("B007"):
        fault_type = "Ball"

    elif name.startswith("IR007"):
        fault_type = "Inner_Race"

    elif name.startswith("OR007@6"):
        fault_type = "Outer_Race"

    else:
        fault_type = "Unknown"

    # Load

    load_hp = int(name.split("_")[-1])

    return fault_type, load_hp


all_rows = []

print("Processing recordings...")
print("=" * 65)

for file in mat_files:

    # Identify recording

    fault_type, load_hp = get_file_info(file.name)

    # Load MATLAB file

    mat = loadmat(file)

    de_keys = [
        key for key in mat.keys()
        if "DE_time" in key
    ]

    rpm_keys = [
        key for key in mat.keys()
        if "RPM" in key
    ]

    if len(de_keys) == 0:
        print("SKIPPED — no DE signal:", file.name)
        continue

    signal = mat[de_keys[0]].flatten()

    original_samples = len(signal)

    # Sampling-rate standardization

    if fault_type == "Normal":

        original_fs = 48000

        # 48 kHz -> 12 kHz
        signal = resample_poly(
            signal,
            up=1,
            down=4
        )

    else:

        original_fs = 12000

    # RPM

    if len(rpm_keys) > 0:

        rpm_value = float(
            mat[rpm_keys[0]].flatten()[0]
        )

    else:

        # Use documented approximate speed
        rpm_value = rpm_by_load[load_hp]

    # Segment into 1-second windows

    num_windows = len(signal) // WINDOW_SIZE

    # Extract features

    for i in range(num_windows):

        start = i * WINDOW_SIZE
        end = start + WINDOW_SIZE

        window = signal[start:end]

        features = extract_features(window)

        features["Fault_Type"] = fault_type
        features["Load_HP"] = load_hp
        features["RPM"] = rpm_value
        features["Source_File"] = file.name
        features["Window"] = i + 1
        features["Sampling_Rate_Hz"] = TARGET_FS

        all_rows.append(features)

    print(
        f"{file.name:15s} | "
        f"{fault_type:11s} | "
        f"Load {load_hp} HP | "
        f"{original_fs:5d} -> {TARGET_FS:5d} Hz | "
        f"{original_samples:6d} samples -> "
        f"{num_windows:2d} windows"
    )


dataset = pd.DataFrame(all_rows)

print("=" * 65)
print("COMPLETE")
print("Dataset shape:", dataset.shape)

In [ ]:
print("DATASET SHAPE")
print(dataset.shape)

print("\nCLASS COUNTS")
print(dataset["Fault_Type"].value_counts())

print("\nCLASS × LOAD COUNTS")
print(
    dataset.groupby(
        ["Fault_Type", "Load_HP"]
    ).size()
)

print("\nMISSING VALUES")
print(dataset.isnull().sum())

print("\nUNKNOWN LABELS")
print(
    dataset[
        dataset["Fault_Type"] == "Unknown"
    ]
)

print("\nWINDOWS PER SOURCE FILE")
print(
    dataset.groupby("Source_File").size()
)

In [ ]:
print("Total rows:", len(dataset))
print("Source files:", dataset["Source_File"].nunique())
print("Unknown labels:", (dataset["Fault_Type"] == "Unknown").sum())
print("Total missing values:", dataset.isnull().sum().sum())

print("\nClass counts:")
print(dataset["Fault_Type"].value_counts())

# Automatic safety checks
assert len(dataset) == 155, "Unexpected number of rows!"
assert dataset["Source_File"].nunique() == 16, "Not all 16 files are present!"
assert (dataset["Fault_Type"] == "Unknown").sum() == 0, "Unknown labels found!"
assert dataset.isnull().sum().sum() == 0, "Missing values found!"

print("\n✓ ALL DATASET CHECKS PASSED")

In [ ]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "bearing_features.csv"

dataset.to_csv(output_file, index=False)

print("Dataset saved successfully.")
print("Location:", output_file.resolve())
print("Shape:", dataset.shape)

In [ ]:
comparison_features = [
    "RMS",
    "Kurtosis",
    "Crest_Factor",
    "Peak_to_Peak"
]

class_summary = (
    dataset
    .groupby("Fault_Type")[comparison_features]
    .agg(["mean", "std"])
)

class_summary

In [ ]:
class_order = [
    "Normal",
    "Ball",
    "Inner_Race",
    "Outer_Race"
]

rms_data = [
    dataset.loc[
        dataset["Fault_Type"] == condition,
        "RMS"
    ].values
    for condition in class_order
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    rms_data,
    tick_labels=class_order
)

plt.xlabel("Bearing Condition")
plt.ylabel("RMS Acceleration (g)")
plt.title("RMS Vibration by Bearing Condition")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
kurtosis_data = [
    dataset.loc[
        dataset["Fault_Type"] == condition,
        "Kurtosis"
    ].values
    for condition in class_order
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    kurtosis_data,
    tick_labels=class_order
)

plt.xlabel("Bearing Condition")
plt.ylabel("Kurtosis")
plt.title("Kurtosis by Bearing Condition")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
crest_data = [
    dataset.loc[
        dataset["Fault_Type"] == condition,
        "Crest_Factor"
    ].values
    for condition in class_order
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    crest_data,
    tick_labels=class_order
)

plt.xlabel("Bearing Condition")
plt.ylabel("Crest Factor")
plt.title("Crest Factor by Bearing Condition")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

threshold_results = []

for test_load in [0, 1, 2, 3]:


    train_data = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    test_data = dataset[
        dataset["Load_HP"] == test_load
    ].copy()


    healthy_train_rms = train_data.loc[
        train_data["Fault_Type"] == "Normal",
        "RMS"
    ]

    healthy_mean = healthy_train_rms.mean()
    healthy_std = healthy_train_rms.std(ddof=1)

    threshold = healthy_mean + 3 * healthy_std

    # Normal = 0
    # Any fault = 1

    y_true = (
        test_data["Fault_Type"] != "Normal"
    ).astype(int)


    y_pred = (
        test_data["RMS"] > threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()


    accuracy = (tp + tn) / (tp + tn + fp + fn)

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0 else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0 else 0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0 else 0
    )

    f1 = (
        2 * precision * sensitivity /
        (precision + sensitivity)
        if (precision + sensitivity) > 0 else 0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0 else 0
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    threshold_results.append({
        "Test_Load_HP": test_load,
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1": f1,
        "False_Alarm_Rate": false_alarm_rate,
        "Balanced_Accuracy": balanced_accuracy
    })


threshold_results = pd.DataFrame(threshold_results)

threshold_results.round(4)

In [ ]:
normal_data = dataset[
    dataset["Fault_Type"] == "Normal"
].copy()

normal_rms_by_load = (
    normal_data
    .groupby("Load_HP")["RMS"]
    .agg(["count", "mean", "std", "min", "max"])
)

normal_rms_by_load.round(6)

In [ ]:
normal_loads = [0, 1, 2, 3]

normal_rms_data = [
    normal_data.loc[
        normal_data["Load_HP"] == load,
        "RMS"
    ].values
    for load in normal_loads
]

plt.figure(figsize=(9, 5))

plt.boxplot(
    normal_rms_data,
    tick_labels=["0 HP", "1 HP", "2 HP", "3 HP"]
)

plt.xlabel("Motor Load")
plt.ylabel("RMS Acceleration (g)")
plt.title("Healthy Bearing RMS Across Motor Loads")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
for filename in [
    "Normal_0.mat",
    "Normal_1.mat",
    "Normal_2.mat",
    "Normal_3.mat"
]:
    
    file = raw_data_dir / filename
    mat = loadmat(file)

    de_keys = [
        key for key in mat.keys()
        if "DE_time" in key
    ]

    signal = mat[de_keys[0]].flatten()

    print(
        f"{filename:15s} | "
        f"DE key: {de_keys[0]:15s} | "
        f"Samples: {len(signal)} | "
        f"First 5: {signal[:5]}"
    )

In [ ]:
mat_1 = loadmat(raw_data_dir / "Normal_1.mat")
mat_2 = loadmat(raw_data_dir / "Normal_2.mat")

key_1 = [k for k in mat_1.keys() if "DE_time" in k][0]
key_2 = [k for k in mat_2.keys() if "DE_time" in k][0]

signal_1 = mat_1[key_1].flatten()
signal_2 = mat_2[key_2].flatten()

print("Normal_1 DE key:", key_1)
print("Normal_2 DE key:", key_2)
print("Same length:", len(signal_1) == len(signal_2))

if len(signal_1) == len(signal_2):
    print("Signals exactly identical:", np.array_equal(signal_1, signal_2))
    print("Maximum difference:", np.max(np.abs(signal_1 - signal_2)))

In [ ]:
mat_2 = loadmat(raw_data_dir / "Normal_2.mat")

de_keys_2 = [
    key for key in mat_2.keys()
    if "DE_time" in key
]

rpm_keys_2 = [
    key for key in mat_2.keys()
    if "RPM" in key
]

print("DE key:", de_keys_2)
print("RPM key:", rpm_keys_2)

signal_2 = mat_2[de_keys_2[0]].flatten()

print("Number of samples:", len(signal_2))
print("First 5 samples:", signal_2[:5])

if rpm_keys_2:
    print("Recorded RPM:", mat_2[rpm_keys_2[0]].flatten()[0])

In [ ]:
signal_2_correct = mat_2["X099_DE_time"].flatten()
signal_2_wrong = mat_2["X098_DE_time"].flatten()

print("Correct key: X099_DE_time")
print("Correct signal samples:", len(signal_2_correct))
print("Correct first 5:", signal_2_correct[:5])

print("\nIncorrect embedded X098 signal:")
print("Samples:", len(signal_2_wrong))
print("First 5:", signal_2_wrong[:5])

print("\nAre X098 and X099 identical?")

if len(signal_2_correct) == len(signal_2_wrong):
    print(np.array_equal(signal_2_correct, signal_2_wrong))
else:
    print("No — different lengths.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import resample_poly

# Explicit DE signal for every recording

expected_de_keys = {
    # Normal
    "Normal_0.mat": "X097_DE_time",
    "Normal_1.mat": "X098_DE_time",
    "Normal_2.mat": "X099_DE_time",   # IMPORTANT CWRU anomaly
    "Normal_3.mat": "X100_DE_time",

    # Inner race fault
    "IR007_0.mat": "X105_DE_time",
    "IR007_1.mat": "X106_DE_time",
    "IR007_2.mat": "X107_DE_time",
    "IR007_3.mat": "X108_DE_time",

    # Ball fault
    "B007_0.mat": "X118_DE_time",
    "B007_1.mat": "X119_DE_time",
    "B007_2.mat": "X120_DE_time",
    "B007_3.mat": "X121_DE_time",

    # Outer race fault
    "OR007@6_0.mat": "X130_DE_time",
    "OR007@6_1.mat": "X131_DE_time",
    "OR007@6_2.mat": "X132_DE_time",
    "OR007@6_3.mat": "X133_DE_time"
}


# Approximate RPM values used only when RPM is absent
rpm_by_load = {
    0: 1797,
    1: 1772,
    2: 1750,
    3: 1730
}


def get_file_info(filename):

    if filename.startswith("Normal"):
        fault_type = "Normal"

    elif filename.startswith("B007"):
        fault_type = "Ball"

    elif filename.startswith("IR007"):
        fault_type = "Inner_Race"

    elif filename.startswith("OR007@6"):
        fault_type = "Outer_Race"

    else:
        raise ValueError(f"Unknown file: {filename}")

    load_hp = int(
        filename.replace(".mat", "").split("_")[-1]
    )

    return fault_type, load_hp


# Process all recordings

all_rows = []

mat_files = sorted(raw_data_dir.glob("*.mat"))

for file in mat_files:

    filename = file.name

    if filename not in expected_de_keys:
        print(f"Skipping unexpected file: {filename}")
        continue

    fault_type, load_hp = get_file_info(filename)

    mat = loadmat(file)

    # Select the CORRECT signal explicitly

    de_key = expected_de_keys[filename]

    if de_key not in mat:
        raise KeyError(
            f"{filename}: expected {de_key}, "
            f"but it was not found."
        )

    signal = mat[de_key].flatten()

    original_samples = len(signal)

    # Sampling-rate standardization
    #
    # Normal recordings = 48 kHz
    # Selected fault recordings = 12 kHz
    #
    # Convert Normal 48 kHz -> 12 kHz

    if fault_type == "Normal":

        signal = resample_poly(
            signal,
            up=1,
            down=4
        )

    fs = 12000

    # RPM

    record_id = de_key.split("_")[0]
    expected_rpm_key = record_id + "RPM"

    if expected_rpm_key in mat:

        rpm = float(
            mat[expected_rpm_key]
            .flatten()[0]
        )

    else:

        rpm = rpm_by_load[load_hp]

    # Create non-overlapping 1-second windows

    window_size = fs

    n_windows = len(signal) // window_size

    for window_index in range(n_windows):

        start = window_index * window_size
        end = start + window_size

        window = signal[start:end]

        features = extract_features(window)

        features["Fault_Type"] = fault_type
        features["Load_HP"] = load_hp
        features["RPM"] = rpm
        features["Source_File"] = filename
        features["Window"] = window_index + 1
        features["Sampling_Rate_Hz"] = fs

        all_rows.append(features)

    print(
        f"{filename:15s} | "
        f"{de_key:12s} | "
        f"original={original_samples:6d} | "
        f"windows={n_windows:2d}"
    )


# Final DataFrame

dataset = pd.DataFrame(all_rows)

print("\nDataset rebuilt.")
print("Shape:", dataset.shape)

In [ ]:
import hashlib


signal_hashes = {}

for filename, de_key in expected_de_keys.items():

    file = raw_data_dir / filename
    mat = loadmat(file)

    signal = mat[de_key].flatten()

    # Create fingerprint of the exact selected signal
    signal_hash = hashlib.sha256(
        signal.tobytes()
    ).hexdigest()

    signal_hashes[filename] = signal_hash


# Look for duplicate fingerprints
hash_to_files = {}

for filename, signal_hash in signal_hashes.items():
    hash_to_files.setdefault(signal_hash, []).append(filename)

duplicate_groups = [
    files
    for files in hash_to_files.values()
    if len(files) > 1
]


print("========== RAW SIGNAL CHECK ==========")

print("Selected source signals:", len(signal_hashes))

if duplicate_groups:
    print("❌ DUPLICATE SIGNALS FOUND:")
    
    for group in duplicate_groups:
        print(group)

else:
    print("✓ All 16 selected source signals are unique.")


print("\n========== DATASET CHECK ==========")

print("Total rows:", len(dataset))
print("Source files:", dataset["Source_File"].nunique())
print("Missing values:", dataset.isnull().sum().sum())

print("\nClass counts:")
print(dataset["Fault_Type"].value_counts())

print("\nWindows by class and load:")
print(
    dataset
    .groupby(["Fault_Type", "Load_HP"])
    .size()
    .unstack(fill_value=0)
)


assert len(duplicate_groups) == 0, \
    "Duplicate selected source signals detected!"

assert len(dataset) == 155, \
    "Unexpected dataset size!"

assert dataset["Source_File"].nunique() == 16, \
    "Expected 16 source files!"

assert dataset.isnull().sum().sum() == 0, \
    "Missing values detected!"

assert set(dataset["Fault_Type"].unique()) == {
    "Normal",
    "Ball",
    "Inner_Race",
    "Outer_Race"
}, "Unexpected fault labels!"

print("\n✓ ALL INTEGRITY CHECKS PASSED")

In [ ]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "bearing_features.csv"

dataset.to_csv(
    output_file,
    index=False
)

print("Corrected dataset saved.")
print("Location:", output_file.resolve())
print("Shape:", dataset.shape)

In [ ]:
normal_data = dataset[
    dataset["Fault_Type"] == "Normal"
].copy()

normal_rms_by_load = (
    normal_data
    .groupby("Load_HP")["RMS"]
    .agg(["count", "mean", "std", "min", "max"])
)

normal_rms_by_load.round(6)

In [ ]:
normal_loads = [0, 1, 2, 3]

normal_rms_data = [
    normal_data.loc[
        normal_data["Load_HP"] == load,
        "RMS"
    ].values
    for load in normal_loads
]

plt.figure(figsize=(9, 5))

plt.boxplot(
    normal_rms_data,
    tick_labels=["0 HP", "1 HP", "2 HP", "3 HP"]
)

plt.xlabel("Motor Load")
plt.ylabel("RMS Acceleration (g)")
plt.title("Healthy Bearing RMS Across Motor Loads")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score
)

# Vibration features only

feature_columns = [
    "RMS",
    "Std_Dev",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

lr_results = []

# Leave-One-Load-Out evaluation

for test_load in [0, 1, 2, 3]:

    train_data = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    test_data = dataset[
        dataset["Load_HP"] == test_load
    ].copy()

    # X = vibration features
    # y = 0 Normal, 1 Fault

    X_train = train_data[feature_columns]
    X_test = test_data[feature_columns]

    y_train = (
        train_data["Fault_Type"] != "Normal"
    ).astype(int)

    y_test = (
        test_data["Fault_Type"] != "Normal"
    ).astype(int)

    # Pipeline:
    # scaling is fitted ONLY on training data

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(X_train, y_train)

    # Predictions

    y_pred = model.predict(X_test)

    y_probability = model.predict_proba(
        X_test
    )[:, 1]

    # Confusion matrix

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    # Metrics

    accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    f1 = (
        2 * precision * sensitivity /
        (precision + sensitivity)
        if (precision + sensitivity) > 0
        else 0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    roc_auc = roc_auc_score(
        y_test,
        y_probability
    )

    # Store fold result

    lr_results.append({
        "Test_Load_HP": test_load,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1": f1,
        "False_Alarm_Rate": false_alarm_rate,
        "Balanced_Accuracy": balanced_accuracy,
        "ROC_AUC": roc_auc
    })


lr_results = pd.DataFrame(lr_results)

lr_results.round(4)

In [ ]:
# RMS THRESHOLD vs LOGISTIC REGRESSION

comparison_results = pd.DataFrame({
    
    "Test_Load_HP":
        threshold_results["Test_Load_HP"],
    
    "RMS_Balanced_Accuracy":
        threshold_results["Balanced_Accuracy"],
    
    "LR_Balanced_Accuracy":
        lr_results["Balanced_Accuracy"],
    
    "RMS_Sensitivity":
        threshold_results["Sensitivity"],
    
    "LR_Sensitivity":
        lr_results["Sensitivity"],
    
    "RMS_Specificity":
        threshold_results["Specificity"],
    
    "LR_Specificity":
        lr_results["Specificity"],
    
    "RMS_False_Alarm_Rate":
        threshold_results["False_Alarm_Rate"],
    
    "LR_False_Alarm_Rate":
        lr_results["False_Alarm_Rate"],
    
    "RMS_F1":
        threshold_results["F1"],
    
    "LR_F1":
        lr_results["F1"]
})

comparison_results.round(4)

In [ ]:
rms_lr_results = []

for test_load in [0, 1, 2, 3]:

    # Leave one load completely out

    train_data = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    test_data = dataset[
        dataset["Load_HP"] == test_load
    ].copy()


    X_train = train_data[["RMS"]]
    X_test = test_data[["RMS"]]

    y_train = (
        train_data["Fault_Type"] != "Normal"
    ).astype(int)

    y_test = (
        test_data["Fault_Type"] != "Normal"
    ).astype(int)

    # Scaling + Logistic Regression
    # Scaling fitted only on training data

    rms_model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    rms_model.fit(X_train, y_train)

    # Predictions

    y_pred = rms_model.predict(X_test)

    y_probability = rms_model.predict_proba(
        X_test
    )[:, 1]

    # Confusion matrix

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    # Metrics

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0 else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0 else 0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0 else 0
    )

    accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
    )

    f1 = (
        2 * precision * sensitivity /
        (precision + sensitivity)
        if (precision + sensitivity) > 0
        else 0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0 else 0
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    roc_auc = roc_auc_score(
        y_test,
        y_probability
    )

    rms_lr_results.append({
        "Test_Load_HP": test_load,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1": f1,
        "False_Alarm_Rate": false_alarm_rate,
        "Balanced_Accuracy": balanced_accuracy,
        "ROC_AUC": roc_auc
    })


rms_lr_results = pd.DataFrame(rms_lr_results)

rms_lr_results.round(4)

In [ ]:
test_load = 1

train_data = dataset[
    dataset["Load_HP"] != test_load
].copy()

test_data = dataset[
    dataset["Load_HP"] == test_load
].copy()

X_train = train_data[["RMS"]]
X_test = test_data[["RMS"]]

y_train = (
    train_data["Fault_Type"] != "Normal"
).astype(int)

y_test = (
    test_data["Fault_Type"] != "Normal"
).astype(int)


# Recreate exact RMS-only model

rms_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

rms_model.fit(X_train, y_train)


# Predictions + fault probabilities

y_pred = rms_model.predict(X_test)

fault_probability = rms_model.predict_proba(
    X_test
)[:, 1]


# Build inspection table

inspection = test_data[
    [
        "Fault_Type",
        "Source_File",
        "Window",
        "RMS"
    ]
].copy()

inspection["True_Label"] = y_test.values
inspection["Predicted_Label"] = y_pred
inspection["Fault_Probability"] = fault_probability


# Show false negatives only

false_negatives = inspection[
    (inspection["True_Label"] == 1) &
    (inspection["Predicted_Label"] == 0)
].copy()

false_negatives.round(6)

In [ ]:
test_load = 1

feature_columns = [
    "RMS",
    "Std_Dev",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

train_data = dataset[
    dataset["Load_HP"] != test_load
].copy()

test_data = dataset[
    dataset["Load_HP"] == test_load
].copy()

y_train = (
    train_data["Fault_Type"] != "Normal"
).astype(int)

y_test = (
    test_data["Fault_Type"] != "Normal"
).astype(int)


# RMS-only model

rms_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

rms_model.fit(
    train_data[["RMS"]],
    y_train
)

rms_probability = rms_model.predict_proba(
    test_data[["RMS"]]
)[:, 1]


# Seven-feature model

full_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

full_model.fit(
    train_data[feature_columns],
    y_train
)

full_probability = full_model.predict_proba(
    test_data[feature_columns]
)[:, 1]


# Comparison table

probability_comparison = test_data[
    [
        "Fault_Type",
        "Source_File",
        "Window",
        "RMS",
        "Kurtosis",
        "Crest_Factor"
    ]
].copy()

probability_comparison["RMS_Probability"] = rms_probability
probability_comparison["Full_Probability"] = full_probability

probability_comparison["RMS_Prediction"] = (
    probability_comparison["RMS_Probability"] >= 0.5
).astype(int)

probability_comparison["Full_Prediction"] = (
    probability_comparison["Full_Probability"] >= 0.5
).astype(int)


# Show only the Ball-fault recording that caused the problem

ball_comparison = probability_comparison[
    probability_comparison["Source_File"] == "B007_1.mat"
].copy()

ball_comparison.round(6)

In [ ]:
# Get fitted scaler and logistic regression model
scaler = full_model.named_steps["scaler"]
classifier = full_model.named_steps["classifier"]

coefficients = classifier.coef_[0]
intercept = classifier.intercept_[0]


coefficient_table = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": coefficients
})

coefficient_table["Abs_Coefficient"] = (
    coefficient_table["Coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "Abs_Coefficient",
    ascending=False
)

print("FULL-MODEL STANDARDIZED COEFFICIENTS")
display(coefficient_table.round(4))


recovered_windows = test_data[
    (test_data["Source_File"] == "B007_1.mat") &
    (test_data["Window"].isin([1, 2, 3, 9]))
].copy()


X_standardized = scaler.transform(
    recovered_windows[feature_columns]
)


#
# Positive contribution -> pushes toward FAULT
# Negative contribution -> pushes toward NORMAL

contributions = pd.DataFrame(
    X_standardized * coefficients,
    columns=feature_columns,
    index=recovered_windows.index
)

contributions.insert(
    0,
    "Window",
    recovered_windows["Window"].values
)


feature_contribution_sum = (
    contributions[feature_columns].sum(axis=1)
)

logit = intercept + feature_contribution_sum

probability = 1 / (1 + np.exp(-logit))

contributions["Intercept"] = intercept
contributions["Total_Logit"] = logit
contributions["Fault_Probability"] = probability


print("\nFEATURE CONTRIBUTIONS FOR RECOVERED WINDOWS")
display(contributions.round(4))

In [ ]:
# RMS vs RMS + ONE ADDITIONAL FEATURE
# Leave-One-Load-Out Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    balanced_accuracy_score,
    f1_score
)

additional_features = [
    "Std_Dev",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

feature_sets = {
    "RMS only": ["RMS"]
}

for feature in additional_features:
    feature_sets[f"RMS + {feature}"] = [
        "RMS",
        feature
    ]


ablation_rows = []

# Test every feature set

for model_name, features in feature_sets.items():

    for test_load in [0, 1, 2, 3]:

        train_data = dataset[
            dataset["Load_HP"] != test_load
        ].copy()

        test_data = dataset[
            dataset["Load_HP"] == test_load
        ].copy()

        X_train = train_data[features]
        X_test = test_data[features]

        y_train = (
            train_data["Fault_Type"] != "Normal"
        ).astype(int)

        y_test = (
            test_data["Fault_Type"] != "Normal"
        ).astype(int)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1]
        ).ravel()

        sensitivity = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0
        )

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        balanced_accuracy = balanced_accuracy_score(
            y_test,
            y_pred
        )

        f1 = f1_score(
            y_test,
            y_pred
        )

        ablation_rows.append({
            "Feature_Set": model_name,
            "Test_Load_HP": test_load,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Balanced_Accuracy": balanced_accuracy,
            "F1": f1
        })


ablation_results = pd.DataFrame(ablation_rows)


# Focus on the difficult 1 HP condition

load1_ablation = (
    ablation_results[
        ablation_results["Test_Load_HP"] == 1
    ]
    .sort_values(
        "Balanced_Accuracy",
        ascending=False
    )
)

print("===== HELD-OUT 1 HP RESULTS =====")
display(load1_ablation.round(4))


# Also summarize performance across all four loads

ablation_summary = (
    ablation_results
    .groupby("Feature_Set")[
        [
            "Sensitivity",
            "Specificity",
            "Balanced_Accuracy",
            "F1"
        ]
    ]
    .mean()
    .sort_values(
        "Balanced_Accuracy",
        ascending=False
    )
)

print("\n===== MEAN ACROSS ALL FOUR HELD-OUT LOADS =====")
display(ablation_summary.round(4))

In [ ]:
feature_columns = [
    "RMS",
    "Std_Dev",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

# Correlation matrix
feature_correlation = dataset[feature_columns].corr()

print("===== FEATURE CORRELATION MATRIX =====")
display(feature_correlation.round(4))


# Specifically inspect RMS vs Standard Deviation

rms_std_check = pd.DataFrame({
    "RMS": dataset["RMS"],
    "Std_Dev": dataset["Std_Dev"]
})

rms_std_check["Difference"] = (
    rms_std_check["RMS"] -
    rms_std_check["Std_Dev"]
)

rms_std_check["Percent_Difference"] = (
    rms_std_check["Difference"].abs()
    / rms_std_check["RMS"]
    * 100
)

print("\n===== RMS vs STD_DEV =====")

print(
    "Correlation:",
    dataset["RMS"].corr(dataset["Std_Dev"])
)

print(
    "Mean absolute difference:",
    rms_std_check["Difference"].abs().mean()
)

print(
    "Maximum absolute difference:",
    rms_std_check["Difference"].abs().max()
)

print(
    "Mean percent difference:",
    rms_std_check["Percent_Difference"].mean()
)

print(
    "Maximum percent difference:",
    rms_std_check["Percent_Difference"].max()
)

In [ ]:
important_features = [
    "RMS",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

for fault_type in [
    "Normal",
    "Ball",
    "Inner_Race",
    "Outer_Race"
]:

    subset = dataset[
        dataset["Fault_Type"] == fault_type
    ]

    print("\n" + "=" * 65)
    print(f"{fault_type.upper()} — WITHIN-CLASS CORRELATIONS")
    print("=" * 65)

    display(
        subset[important_features]
        .corr()
        .round(3)
    )

In [ ]:
# Leave-One-Load-Out Evaluation

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

# Features
# Std_Dev deliberately excluded because it is nearly
# perfectly redundant with RMS

multiclass_features = [
    "RMS",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

class_order = [
    "Normal",
    "Ball",
    "Inner_Race",
    "Outer_Race"
]

multiclass_lr_results = []
all_true = []
all_pred = []

# Leave-One-Load-Out

for test_load in sorted(dataset["Load_HP"].unique()):

    train = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    test = dataset[
        dataset["Load_HP"] == test_load
    ].copy()

    X_train = train[multiclass_features]
    y_train = train["Fault_Type"]

    X_test = test[multiclass_features]
    y_test = test["Fault_Type"]

    # Scaling is learned ONLY from training data
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            max_iter=5000
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Store predictions for overall confusion matrix
    all_true.extend(y_test.tolist())
    all_pred.extend(y_pred.tolist())

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    # Number correct
    correct = (y_pred == y_test).sum()
    total = len(y_test)

    multiclass_lr_results.append({
        "Test_Load_HP": test_load,
        "Correct": correct,
        "Total": total,
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_accuracy,
        "Macro_F1": macro_f1
    })


# Results table

multiclass_lr_results = pd.DataFrame(
    multiclass_lr_results
)

print("\n===== MULTICLASS LOGISTIC REGRESSION =====")

display(
    multiclass_lr_results.round(4)
)


# Overall confusion matrix

overall_cm = confusion_matrix(
    all_true,
    all_pred,
    labels=class_order
)

overall_cm_df = pd.DataFrame(
    overall_cm,
    index=[f"True_{x}" for x in class_order],
    columns=[f"Pred_{x}" for x in class_order]
)

print("\n===== AGGREGATED CONFUSION MATRIX =====")

display(overall_cm_df)

In [ ]:
# Train on 0, 1, 2 HP
train = dataset[
    dataset["Load_HP"] != 3
].copy()

# Test ONLY on unseen 3 HP
test = dataset[
    dataset["Load_HP"] == 3
].copy()


X_train = train[multiclass_features]
y_train = train["Fault_Type"]

X_test = test[multiclass_features]
y_test = test["Fault_Type"]


# Train Logistic Regression

model_3hp = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=5000
    ))
])

model_3hp.fit(X_train, y_train)


# Predictions + probabilities

y_pred = model_3hp.predict(X_test)

probabilities = model_3hp.predict_proba(X_test)

classes = model_3hp.named_steps[
    "logistic"
].classes_


# Build inspection table

inspection_3hp = test[
    [
        "Fault_Type",
        "Source_File",
        "Window",
        "RMS",
        "Peak",
        "Peak_to_Peak",
        "Skewness",
        "Kurtosis",
        "Crest_Factor"
    ]
].copy()

inspection_3hp["Prediction"] = y_pred

inspection_3hp["Correct"] = (
    inspection_3hp["Fault_Type"]
    == inspection_3hp["Prediction"]
)


# Add probability for every class
for i, class_name in enumerate(classes):

    inspection_3hp[
        f"P_{class_name}"
    ] = probabilities[:, i]


# Show ONLY errors

errors_3hp = inspection_3hp[
    inspection_3hp["Correct"] == False
].copy()

print(
    "Number of 3 HP misclassifications:",
    len(errors_3hp)
)

display(
    errors_3hp.round(4)
)

In [ ]:
# Compare class means at each motor load

shift_features = [
    "RMS",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

feature_shift = (
    dataset
    .groupby(["Fault_Type", "Load_HP"])[shift_features]
    .mean()
    .round(4)
)

print("===== MEAN FEATURES BY CLASS AND LOAD =====")

display(feature_shift)

In [ ]:
# Class-specific feature contributions to LR logits

# Get fitted scaler + logistic regression model
scaler = model_3hp.named_steps["scaler"]
lr = model_3hp.named_steps["logistic"]

# Select ONLY the problematic IR007_3 windows
ir3 = test[
    test["Source_File"] == "IR007_3.mat"
].copy()

X_ir3 = ir3[multiclass_features]

# Standardize using TRAINING statistics only
X_ir3_scaled = scaler.transform(X_ir3)

# Average standardized feature vector across the 10 windows
mean_scaled = X_ir3_scaled.mean(axis=0)

# Contribution = standardized feature × class coefficient

contribution_rows = []

for class_index, class_name in enumerate(lr.classes_):

    coefficients = lr.coef_[class_index]

    contributions = (
        mean_scaled * coefficients
    )

    row = {
        "Class": class_name,
        "Intercept": lr.intercept_[class_index]
    }

    for feature, contribution in zip(
        multiclass_features,
        contributions
    ):
        row[feature] = contribution

    row["Total_Logit"] = (
        lr.intercept_[class_index]
        + contributions.sum()
    )

    contribution_rows.append(row)


contribution_table = pd.DataFrame(
    contribution_rows
).set_index("Class")


print(
    "===== MEAN LOGIT CONTRIBUTIONS FOR IR007_3 ====="
)

display(
    contribution_table.round(4)
)


# Also show coefficients themselves

coefficient_table = pd.DataFrame(
    lr.coef_,
    index=lr.classes_,
    columns=multiclass_features
)

print(
    "\n===== MULTICLASS LR STANDARDIZED COEFFICIENTS ====="
)

display(
    coefficient_table.round(4)
)

In [ ]:
# Leave-One-Load-Out Evaluation

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

rf_results = []

rf_all_true = []
rf_all_pred = []

# Same six features used for Logistic Regression

rf_features = [
    "RMS",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]


# Leave-One-Load-Out

for test_load in sorted(dataset["Load_HP"].unique()):

    train = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    test = dataset[
        dataset["Load_HP"] == test_load
    ].copy()

    X_train = train[rf_features]
    y_train = train["Fault_Type"]

    X_test = test[rf_features]
    y_test = test["Fault_Type"]


    # Random Forest

    rf_model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        X_train,
        y_train
    )

    y_pred = rf_model.predict(X_test)


    # Store predictions

    rf_all_true.extend(
        y_test.tolist()
    )

    rf_all_pred.extend(
        y_pred.tolist()
    )


    # Metrics

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    correct = (
        y_pred == y_test
    ).sum()

    total = len(y_test)


    rf_results.append({
        "Test_Load_HP": test_load,
        "Correct": correct,
        "Total": total,
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_accuracy,
        "Macro_F1": macro_f1
    })


# Results table

rf_results = pd.DataFrame(
    rf_results
)

print(
    "\n===== MULTICLASS RANDOM FOREST ====="
)

display(
    rf_results.round(4)
)


# Aggregated confusion matrix

rf_cm = confusion_matrix(
    rf_all_true,
    rf_all_pred,
    labels=class_order
)

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=[
        f"True_{x}"
        for x in class_order
    ],
    columns=[
        f"Pred_{x}"
        for x in class_order
    ]
)

print(
    "\n===== RANDOM FOREST AGGREGATED CONFUSION MATRIX ====="
)

display(
    rf_cm_df
)

In [ ]:
# Calculated separately for each held-out-load fold

rf_importance_rows = []

for test_load in sorted(dataset["Load_HP"].unique()):

    train = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    X_train = train[rf_features]
    y_train = train["Fault_Type"]

    rf_model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    for feature, importance in zip(
        rf_features,
        rf_model.feature_importances_
    ):

        rf_importance_rows.append({
            "Test_Load_HP": test_load,
            "Feature": feature,
            "Importance": importance
        })


rf_feature_importance = pd.DataFrame(
    rf_importance_rows
)


# Importance for every LOLO fold

importance_by_load = (
    rf_feature_importance
    .pivot(
        index="Feature",
        columns="Test_Load_HP",
        values="Importance"
    )
)

importance_by_load.columns = [
    f"Holdout_{x}HP"
    for x in importance_by_load.columns
]


# Mean + variability across folds

importance_summary = (
    rf_feature_importance
    .groupby("Feature")["Importance"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)


print("===== RF FEATURE IMPORTANCE BY HELD-OUT LOAD =====")
display(importance_by_load.round(4))

print("\n===== RF FEATURE IMPORTANCE SUMMARY =====")
display(importance_summary.round(4))

In [ ]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

# Same feature set used for LR and Random Forest
# Std_Dev remains excluded because it is almost identical to RMS

xgb_features = [
    "RMS",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]

# Fixed class ordering
class_order = [
    "Normal",
    "Ball",
    "Inner_Race",
    "Outer_Race"
]

class_to_number = {
    "Normal": 0,
    "Ball": 1,
    "Inner_Race": 2,
    "Outer_Race": 3
}

number_to_class = {
    value: key
    for key, value in class_to_number.items()
}

# Storage

xgb_results = []
xgb_confusion_matrices = []

# Leave-one-load-out

for test_load in [0, 1, 2, 3]:

    # Training = other 3 loads
    train_data = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    # Testing = completely held-out load
    test_data = dataset[
        dataset["Load_HP"] == test_load
    ].copy()

    X_train = train_data[xgb_features]
    X_test = test_data[xgb_features]

    y_train = (
        train_data["Fault_Type"]
        .map(class_to_number)
        .values
    )

    y_test = (
        test_data["Fault_Type"]
        .map(class_to_number)
        .values
    )

    # XGBoost model
    # Hyperparameters kept fixed across ALL held-out loads

    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=4,

        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,

        subsample=0.8,
        colsample_bytree=0.8,

        reg_lambda=1.0,

        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1
    )

    # Train only on the training loads
    xgb_model.fit(
        X_train,
        y_train
    )

    # Predict held-out load

    y_pred = xgb_model.predict(X_test)

    # Metrics

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    correct = (y_test == y_pred).sum()

    # Store fold results

    xgb_results.append({
        "Test_Load_HP": test_load,
        "Correct": correct,
        "Total": len(y_test),
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_accuracy,
        "Macro_F1": macro_f1
    })

    # Confusion matrix for this fold
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3]
    )

    xgb_confusion_matrices.append(cm)


xgb_results = pd.DataFrame(xgb_results)

print("\n===== MULTICLASS XGBOOST =====")

display(
    xgb_results.round(4)
)


xgb_total_cm = np.sum(
    xgb_confusion_matrices,
    axis=0
)

xgb_cm_df = pd.DataFrame(
    xgb_total_cm,
    index=[
        f"True_{x}"
        for x in class_order
    ],
    columns=[
        f"Pred_{x}"
        for x in class_order
    ]
)

print("\n===== XGBOOST AGGREGATED CONFUSION MATRIX =====")

display(xgb_cm_df)

In [ ]:
# Leave-One-Load-Out Results

final_model_comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],

    # Mean accuracy across the four held-out-load folds
    "Mean_Accuracy": [
        (1.000 + 0.975 + 1.000 + 0.750) / 4,
        (1.000 + 1.000 + 1.000 + 1.000) / 4,
        (1.000 + 1.000 + 0.975 + 0.925) / 4
    ],

    # Mean balanced accuracy across folds
    "Mean_Balanced_Accuracy": [
        (1.000 + 0.975 + 1.000 + 0.750) / 4,
        1.000,
        (1.000 + 1.000 + 0.975 + 0.925) / 4
    ],

    # Mean Macro-F1 across folds
    "Mean_Macro_F1": [
        (1.0000 + 0.9749 + 1.0000 + 0.6605) / 4,
        1.0000,
        (1.0000 + 1.0000 + 0.9749 + 0.9213) / 4
    ],

    # Total correctly classified windows
    "Correct_Windows": [
        144,
        155,
        151
    ],

    "Total_Windows": [
        155,
        155,
        155
    ]
})

final_model_comparison["Overall_Accuracy"] = (
    final_model_comparison["Correct_Windows"]
    / final_model_comparison["Total_Windows"]
)

final_model_comparison = (
    final_model_comparison
    .sort_values(
        "Mean_Balanced_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    final_model_comparison.round(4)
)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Get balanced accuracy in load order
lr_balanced = (
    multiclass_lr_results
    .sort_values("Test_Load_HP")["Balanced_Accuracy"]
    .values
)

rf_balanced = (
    rf_results
    .sort_values("Test_Load_HP")["Balanced_Accuracy"]
    .values
)

xgb_balanced = (
    xgb_results
    .sort_values("Test_Load_HP")["Balanced_Accuracy"]
    .values
)

loads = [0, 1, 2, 3]

# Print values first so we can verify the graph

print("Logistic Regression:", lr_balanced)
print("Random Forest:      ", rf_balanced)
print("XGBoost:            ", xgb_balanced)

# Plot

plt.figure(figsize=(10, 6))

plt.plot(
    loads,
    lr_balanced,
    marker="o",
    linewidth=2,
    label="Logistic Regression"
)

plt.plot(
    loads,
    rf_balanced,
    marker="s",
    linewidth=2,
    label="Random Forest"
)

plt.plot(
    loads,
    xgb_balanced,
    marker="^",
    linewidth=2,
    label="XGBoost"
)

plt.xlabel("Held-Out Motor Load (HP)")
plt.ylabel("Balanced Accuracy")
plt.title(
    "Multiclass Fault Diagnosis Performance Across Held-Out Motor Loads"
)

plt.xticks(loads)
plt.ylim(0.70, 1.02)
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

# Save publication-quality figure

figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)

figure_path = figures_dir / "multiclass_model_comparison_by_load.png"

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved to:", figure_path)

In [ ]:
# Leave-One-Load-Out Evaluation

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# Binary target
# 0 = Normal
# 1 = Fault

dataset["Binary_Label"] = np.where(
    dataset["Fault_Type"] == "Normal",
    0,
    1
)


# Same full time-domain feature set used by our full LR model
binary_features = [
    "RMS",
    "Std_Dev",
    "Peak",
    "Peak_to_Peak",
    "Skewness",
    "Kurtosis",
    "Crest_Factor"
]


rf_binary_rows = []
xgb_binary_rows = []


# Leave one motor load out

for test_load in [0, 1, 2, 3]:

    train = dataset[
        dataset["Load_HP"] != test_load
    ].copy()

    test = dataset[
        dataset["Load_HP"] == test_load
    ].copy()

    X_train = train[binary_features]
    y_train = train["Binary_Label"]

    X_test = test[binary_features]
    y_test = test["Binary_Label"]


    rf = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    rf_pred = rf.predict(X_test)
    rf_prob = rf.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        rf_pred,
        labels=[0, 1]
    ).ravel()

    rf_binary_rows.append({
        "Test_Load_HP": test_load,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Sensitivity": recall_score(y_test, rf_pred),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_test, rf_pred),
        "F1": f1_score(y_test, rf_pred),
        "False_Alarm_Rate": fp / (fp + tn),
        "Balanced_Accuracy":
            balanced_accuracy_score(y_test, rf_pred),
        "ROC_AUC": roc_auc_score(y_test, rf_prob)
    })


    xgb_binary = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    xgb_binary.fit(X_train, y_train)

    xgb_pred = xgb_binary.predict(X_test)
    xgb_prob = xgb_binary.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        xgb_pred,
        labels=[0, 1]
    ).ravel()

    xgb_binary_rows.append({
        "Test_Load_HP": test_load,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy_score(y_test, xgb_pred),
        "Sensitivity": recall_score(y_test, xgb_pred),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_test, xgb_pred),
        "F1": f1_score(y_test, xgb_pred),
        "False_Alarm_Rate": fp / (fp + tn),
        "Balanced_Accuracy":
            balanced_accuracy_score(y_test, xgb_pred),
        "ROC_AUC": roc_auc_score(y_test, xgb_prob)
    })


# Results

rf_binary_results = pd.DataFrame(rf_binary_rows)
xgb_binary_results = pd.DataFrame(xgb_binary_rows)


print("\n===== BINARY RANDOM FOREST =====")
display(rf_binary_results.round(4))


print("\n===== BINARY XGBOOST =====")
display(xgb_binary_results.round(4))

In [ ]:
# RMS Threshold vs LR vs RF vs XGBoost

binary_models = {
    "RMS Threshold": threshold_results,
    "Logistic Regression": lr_results,
    "Random Forest": rf_binary_results,
    "XGBoost": xgb_binary_results
}

binary_summary_rows = []

for model_name, results in binary_models.items():

    # Aggregate confusion matrix across all held-out loads

    TN = results["TN"].sum()
    FP = results["FP"].sum()
    FN = results["FN"].sum()
    TP = results["TP"].sum()

    total = TN + FP + FN + TP

    overall_accuracy = (TP + TN) / total

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0 else np.nan
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0 else np.nan
    )

    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0 else np.nan
    )

    f1 = (
        2 * precision * sensitivity
        / (precision + sensitivity)
        if (precision + sensitivity) > 0 else np.nan
    )

    false_alarm_rate = (
        FP / (FP + TN)
        if (FP + TN) > 0 else np.nan
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    # Store result

    binary_summary_rows.append({

        "Model": model_name,

        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP,

        "Overall_Accuracy": overall_accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1": f1,
        "False_Alarm_Rate": false_alarm_rate,
        "Balanced_Accuracy": balanced_accuracy,

        # Mean across the four LOLO folds
        "Mean_Fold_Balanced_Accuracy":
            results["Balanced_Accuracy"].mean()
    })


binary_final_summary = pd.DataFrame(binary_summary_rows)

print(
    "\n===== FINAL BINARY FAULT-DETECTION COMPARISON ====="
)

display(
    binary_final_summary.round(4)
)

In [ ]:
# Performance Across Held-Out Motor Loads

import matplotlib.pyplot as plt
from pathlib import Path

loads = [0, 1, 2, 3]

# Sort all results by held-out load
rms_plot = threshold_results.sort_values("Test_Load_HP")
lr_plot = lr_results.sort_values("Test_Load_HP")
rf_plot = rf_binary_results.sort_values("Test_Load_HP")
xgb_plot = xgb_binary_results.sort_values("Test_Load_HP")

plt.figure(figsize=(10, 6))

plt.plot(
    loads,
    rms_plot["Balanced_Accuracy"],
    marker="o",
    linewidth=2,
    label="RMS Threshold"
)

plt.plot(
    loads,
    lr_plot["Balanced_Accuracy"],
    marker="s",
    linewidth=2,
    label="Logistic Regression"
)

plt.plot(
    loads,
    rf_plot["Balanced_Accuracy"],
    marker="^",
    linewidth=2,
    label="Random Forest"
)

plt.plot(
    loads,
    xgb_plot["Balanced_Accuracy"],
    marker="D",
    linewidth=2,
    label="XGBoost"
)

plt.xlabel("Held-Out Motor Load (HP)")
plt.ylabel("Balanced Accuracy")

plt.title(
    "Binary Fault Detection Performance Across Held-Out Motor Loads"
)

plt.xticks(loads)
plt.ylim(0.45, 1.02)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

# Save figure
figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)

figure_path = (
    figures_dir /
    "binary_fault_detection_balanced_accuracy_by_load.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figure saved to:", figure_path)

In [ ]:
from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)


dataset.to_csv(
    results_dir / "phase1_feature_dataset.csv",
    index=False
)


threshold_results.to_csv(
    results_dir / "binary_rms_threshold_results.csv",
    index=False
)

lr_results.to_csv(
    results_dir / "binary_logistic_regression_results.csv",
    index=False
)

rf_binary_results.to_csv(
    results_dir / "binary_random_forest_results.csv",
    index=False
)

xgb_binary_results.to_csv(
    results_dir / "binary_xgboost_results.csv",
    index=False
)

binary_final_summary.to_csv(
    results_dir / "binary_final_comparison.csv",
    index=False
)


multiclass_lr_results.to_csv(
    results_dir / "multiclass_logistic_regression_results.csv",
    index=False
)

rf_results.to_csv(
    results_dir / "multiclass_random_forest_results.csv",
    index=False
)

xgb_results.to_csv(
    results_dir / "multiclass_xgboost_results.csv",
    index=False
)

final_model_comparison.to_csv(
    results_dir / "multiclass_final_comparison.csv",
    index=False
)


if "overall_cm_df" in globals():
    overall_cm_df.to_csv(
        results_dir / "multiclass_lr_confusion_matrix.csv"
    )

if "rf_cm_df" in globals():
    rf_cm_df.to_csv(
        results_dir / "multiclass_rf_confusion_matrix.csv"
    )

if "xgb_cm_df" in globals():
    xgb_cm_df.to_csv(
        results_dir / "multiclass_xgb_confusion_matrix.csv"
    )


# Confirm files

print("✓ PHASE I RESULTS SAVED")
print("\nResults folder:")
print(results_dir.resolve())

print("\nSaved files:")

for file in sorted(results_dir.glob("*.csv")):
    print("  ✓", file.name)